# Laya Integrated Memory V2.4 — Bi-Gated-Delta-Lite Decision Experiment
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_Integrated_Memory_V24_BiGatedDeltaLite_Decision_Colab.ipynb)


Controlled successor to Laya_Integrated_Memory_V23_Decision_Colab.ipynb.

This notebook measures two stages independently:

1. **V2.4 core** — every ModernBERT full-attention layer is replaced by a bidirectional Gated-Delta-Lite mixer while reusing the source Q/K/V and output projections.
2. **V2.4 compact** — optionally replaces the large sequence-wide decision Transformer with a marker-only 256-d head operating only on CLS + option markers.

It reports teacher/student agreement, JS divergence, latency, throughput-related timing, parameter count, and CUDA activation peak. It does not overwrite V2.3.

Design: no global T×T matrix, no V2.3 96-d softmax/ELU feature maps, forward + reverse Delta scans for bidirectional context, frozen source Wqkv/Wo, and FLA/Triton chunk Gated-Delta kernels.


In [1]:
#@title 1. Install
%pip -q install -U "transformers>=4.45" "datasets>=2.20" "huggingface_hub>=0.25" "safetensors>=0.4"
%pip -q install -U "flash-linear-attention[cuda]"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 65.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.2/819.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 12.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
#@title 2. Imports, configuration, source teacher/student
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from fla.ops.gated_delta_rule import chunk_gated_delta_rule

from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    full_attention_indices, normalize_question,
    _apply_modernbert_rope, _valid_tokens,
)

SEED = 2026
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}

LOCAL_STEPS = 120 #@param {type:"integer"}
GLOBAL_STEPS = 400 #@param {type:"integer"}
TRAIN_CASES = 700 #@param {type:"integer"}
VAL_CASES = 160 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}

USE_BIDIRECTIONAL = True #@param {type:"boolean"}
ALLOW_NEG_EIGVAL = False #@param {type:"boolean"}
LOCAL_LR = 0.003 #@param {type:"number"}
GLOBAL_DELTA_LR = 0.15 #@param {type:"number"}
GLOBAL_HEAD_LR = 0.0002 #@param {type:"number"}

RUN_COMPACT_HEAD = True #@param {type:"boolean"}
COMPACT_DIM = 256 #@param {type:"integer"}
COMPACT_LAYERS = 2 #@param {type:"integer"}
COMPACT_STEPS = 300 #@param {type:"integer"}
COMPACT_JOINT_STEPS = 150 #@param {type:"integer"}

OUTPUT_DIR = Path("/content/Laya_Integrated_Memory_V24_BiGatedDeltaLite")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required for the FLA/Triton performance path.")

device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_full, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))

teacher.to(device).eval().requires_grad_(False)
student_full.to(device).eval().requires_grad_(False)

full_layers = full_attention_indices(student_full)
print("device:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "| amp:", amp_dtype)
print("source:", SOURCE_MODEL)
print("full-attention layers:", full_layers, "| count:", len(full_layers))
assert full_layers


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

device: Tesla T4
torch: 2.11.0+cu128 | amp: torch.bfloat16
source: convaiinnovations/laya-typed-decisions
full-attention layers: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27] | count: 10


In [3]:
#@title 3. V2.4 Bi-Gated-Delta-Lite replacement
class BiGatedDeltaLiteAttention(nn.Module):
    architecture = "integrated_memory_v24_bigated_delta_lite"

    def __init__(self, original: nn.Module, bidirectional: bool = True):
        super().__init__()
        self.config = original.config
        self.layer_idx = int(getattr(original, "layer_idx", -1))
        self.hidden_size = int(self.config.hidden_size)
        self.num_heads = int(self.config.num_attention_heads)
        self.head_dim = int(getattr(original, "head_dim", self.hidden_size // self.num_heads))
        self.bidirectional = bool(bidirectional)

        self.Wqkv = copy.deepcopy(original.Wqkv)
        self.Wo = copy.deepcopy(original.Wo)
        self.out_drop = copy.deepcopy(original.out_drop)
        for module in (self.Wqkv, self.Wo):
            for p in module.parameters():
                p.requires_grad = False

        # Tiny dynamic gate: hidden -> [decay per head | beta per head].
        self.gate_proj = nn.Linear(self.hidden_size, 2 * self.num_heads, bias=True)
        nn.init.zeros_(self.gate_proj.weight)
        with torch.no_grad():
            self.gate_proj.bias[:self.num_heads].zero_()
            self.gate_proj.bias[self.num_heads:].fill_(-1.5)

        # Cheap diagonal adaptation instead of new dense Q/K projections.
        self.q_log_scale = nn.Parameter(torch.zeros(self.num_heads, self.head_dim))
        self.k_log_scale = nn.Parameter(torch.zeros(self.num_heads, self.head_dim))

        gen = torch.Generator().manual_seed(SEED + max(0, self.layer_idx))
        A = torch.empty(self.num_heads).uniform_(0.5, 8.0, generator=gen)
        self.A_log = nn.Parameter(torch.log(A))
        dt = torch.exp(
            torch.empty(self.num_heads).uniform_(math.log(0.001), math.log(0.05), generator=gen)
        ).clamp_min(1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))

        n_routes = 2 if self.bidirectional else 1
        self.mix_logits = nn.Parameter(torch.zeros(self.num_heads, n_routes))
        self.output_gain = nn.Parameter(torch.zeros(self.num_heads))

    def trainable_core_parameters(self):
        frozen = {id(p) for p in self.Wqkv.parameters()} | {id(p) for p in self.Wo.parameters()}
        return [p for p in self.parameters() if p.requires_grad and id(p) not in frozen]

    def _qkv(self, hidden_states, position_embeddings=None):
        b, t, _ = hidden_states.shape
        qkv = self.Wqkv(hidden_states).view(b, t, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        q, k = _apply_modernbert_rope(q, k, position_embeddings)
        qs = self.q_log_scale.clamp(-1, 1).exp()[None, :, None, :]
        ks = self.k_log_scale.clamp(-1, 1).exp()[None, :, None, :]
        return q * qs, k * ks, v

    def _gate_values(self, hidden_states, valid):
        # Materialize tiny gates so padding is exactly neutral in both scan directions.
        raw = self.gate_proj(hidden_states)
        a_raw, beta_raw = raw.chunk(2, dim=-1)
        rate = self.A_log.float().clamp(-4, 4).exp()[None, None, :] * F.softplus(
            a_raw + self.dt_bias.float()[None, None, :]
        )
        g = -rate.clamp_max(1.0)
        beta = torch.sigmoid(beta_raw)
        m = valid[:, :, None].float()
        return (g * m).to(hidden_states.dtype), (beta * m).to(hidden_states.dtype)

    def _run(self, q, k, v, g, beta, valid):
        m = valid[:, :, None, None].to(q.dtype)
        out, _ = chunk_gated_delta_rule(
            q=q * m,
            k=k * m,
            v=v * m,
            g=g,
            beta=beta,
            output_final_state=False,
            use_qk_l2norm_in_kernel=True,
            use_beta_sigmoid_in_kernel=False,
            allow_neg_eigval=ALLOW_NEG_EIGVAL,
            state_v_first=True,
        )
        return out * m

    def forward(self, hidden_states, position_embeddings=None, attention_mask=None, **_):
        q, k, v = self._qkv(hidden_states, position_embeddings)
        valid = _valid_tokens(attention_mask, hidden_states)

        # FLA layout is [B,T,H,D].
        q = q.transpose(1, 2).contiguous()
        k = k.transpose(1, 2).contiguous()
        v = v.transpose(1, 2).contiguous()
        g, beta = self._gate_values(hidden_states, valid)

        fwd = self._run(q, k, v, g, beta, valid)
        if self.bidirectional:
            rv = torch.flip(valid, dims=[1])
            bwd = self._run(
                torch.flip(q, dims=[1]).contiguous(),
                torch.flip(k, dims=[1]).contiguous(),
                torch.flip(v, dims=[1]).contiguous(),
                torch.flip(g, dims=[1]).contiguous(),
                torch.flip(beta, dims=[1]).contiguous(),
                rv,
            )
            bwd = torch.flip(bwd, dims=[1])
            mix = torch.softmax(self.mix_logits.float(), dim=-1)
            core = (
                fwd.float() * mix[:, 0][None, None, :, None]
                + bwd.float() * mix[:, 1][None, None, :, None]
            )
        else:
            core = fwd.float()

        gain = self.output_gain.float().clamp(-2, 2).exp()
        core = core * gain[None, None, :, None]
        core = core * valid[:, :, None, None].float()
        b, t, _, _ = core.shape
        flat = core.reshape(b, t, self.hidden_size).to(hidden_states.dtype)
        return self.out_drop(self.Wo(flat)), None


def v24_core_parameters(model):
    seen, out = set(), []
    for module in model.modules():
        if isinstance(module, BiGatedDeltaLiteAttention):
            for p in module.trainable_core_parameters():
                if id(p) not in seen:
                    seen.add(id(p)); out.append(p)
    return out

def v24_indices(model):
    return [
        i for i, layer in enumerate(model.encoder.layers)
        if isinstance(layer.attn, BiGatedDeltaLiteAttention)
    ]

print("BiGatedDeltaLiteAttention ready.")


BiGatedDeltaLiteAttention ready.


In [4]:
#@title 4. Build typed-decision train/validation data
def _jsonish(x):
    if isinstance(x, str):
        try: return json.loads(x)
        except Exception: return x
    return x

def make_items(split, limit, seed):
    ds = list(load_dataset("LocalLLaMA/typed-decisions", "all", split=split))
    rng = random.Random(seed); rng.shuffle(ds); ds = ds[:min(limit, len(ds))]
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in ds:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "), add_special_tokens=False
        )["input_ids"]
        for qid, qdef in questions.items():
            try:
                q = normalize_question(qdef)
                ids, markers = build_sequence(
                    tokenizer, state, q,
                    max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                    head_max_len=head_max_len,
                    truncate_left=isinstance(state, list),
                    state_ids=state_ids,
                )
                if len(markers) >= 2:
                    out.append({
                        "ids": ids, "markers": markers,
                        "qtype": QTYPES[q["t"]], "qid": qid,
                    })
            except Exception:
                pass
    rng.shuffle(out)
    return out

train_items = make_items("train", TRAIN_CASES, SEED)
val_items = make_items("test", VAL_CASES, SEED + 1)
print("train items:", len(train_items), "| validation items:", len(val_items))
assert len(train_items) >= 64 and len(val_items) >= 32

def batch(items, offset, n=BATCH):
    selected = [items[(offset+i) % len(items)] for i in range(n)]
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k:v.to(device) for k,v in b.items()}

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove(); h2.remove()
    return got

def local_loss(pred, target, valid):
    m = valid[:, :, None].float()
    mse = (((pred.float()-target.float())**2)*m).sum() / (m.sum()*pred.shape[-1]).clamp_min(1)
    power = ((target.float()**2)*m).sum() / (m.sum()*target.shape[-1]).clamp_min(1)
    nmse = mse / power.clamp_min(1e-8)
    cos = F.cosine_similarity(pred.float()[valid], target.float()[valid], dim=-1).mean()
    return nmse + 1.10*(1-cos), nmse, cos


README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

all/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  599kB            

all/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

train items: 3500 | validation items: 800


In [5]:
#@title 5. Stage A — progressive local transfer
local_report = {}

for layer_idx in full_layers:
    print(f"\n=== full-attention layer {layer_idx} ===")
    original = student_full.encoder.layers[layer_idx].attn
    repl = BiGatedDeltaLiteAttention(original, bidirectional=USE_BIDIRECTIONAL).to(device)
    params = repl.trainable_core_parameters()
    opt = torch.optim.AdamW(params, lr=LOCAL_LR, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(1, LOCAL_STEPS), eta_min=LOCAL_LR * 0.08
    )
    best = None; best_score = float("inf")

    probe_b = batch(val_items, layer_idx*11, min(5, BATCH+1))
    probe = attn_io(student_full, layer_idx, probe_b)
    probe_valid = probe_b["attention_mask"].bool()

    for step in range(LOCAL_STEPS):
        b = batch(train_items, step*BATCH + layer_idx*17)
        target = attn_io(student_full, layer_idx, b)
        valid = b["attention_mask"].bool()

        repl.train(); repl.out_drop.eval(); opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            pred, _ = repl(
                target["x"],
                position_embeddings=target["pos"],
                attention_mask=target["mask"],
            )
            loss, nmse, cos = local_loss(pred, target["y"], valid)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite local loss at layer {layer_idx}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); sched.step()

        if step == 0 or (step+1) % max(20, LOCAL_STEPS//4) == 0 or step+1 == LOCAL_STEPS:
            repl.eval()
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
                pp, _ = repl(
                    probe["x"],
                    position_embeddings=probe["pos"],
                    attention_mask=probe["mask"],
                )
                ploss, pnmse, pcos = local_loss(pp, probe["y"], probe_valid)
            score = float(ploss)
            if score < best_score:
                best_score = score
                best = {k:v.detach().cpu().clone() for k,v in repl.state_dict().items()}
            print(
                f"step {step+1:03d}/{LOCAL_STEPS} "
                f"nmse={float(pnmse):.4f} cos={float(pcos):.4f}"
            )

    if best is not None:
        repl.load_state_dict(best)
    repl.eval()
    student_full.encoder.layers[layer_idx].attn = repl
    local_report[str(layer_idx)] = {"best_local_score": best_score}
    torch.cuda.empty_cache()

print("\nconverted:", v24_indices(student_full))
assert v24_indices(student_full) == full_layers
print("V2.4 new core params:", f"{sum(p.numel() for p in v24_core_parameters(student_full)):,}")



=== full-attention layer 0 ===
step 001/120 nmse=0.9420 cos=0.4833
step 030/120 nmse=0.8808 cos=0.7149
step 060/120 nmse=0.8525 cos=0.7452
step 090/120 nmse=0.8370 cos=0.7536
step 120/120 nmse=0.8314 cos=0.7561

=== full-attention layer 3 ===
step 001/120 nmse=0.9431 cos=0.4610


KeyboardInterrupt: 

In [ ]:
#@title 6. Stage B — end-to-end recovery with original decision head
for p in student_full.parameters():
    p.requires_grad = False

delta_params = v24_core_parameters(student_full)
for p in delta_params:
    p.requires_grad = True

head_params = []
for module in (student_full.head, student_full.type_emb, student_full.scorer, student_full.act_head):
    if module is not None:
        for p in module.parameters():
            p.requires_grad = True
            head_params.append(p)

opt = torch.optim.AdamW([
    {"params": delta_params, "lr": GLOBAL_DELTA_LR, "weight_decay": 1e-3},
    {"params": head_params, "lr": GLOBAL_HEAD_LR, "weight_decay": 1e-3},
])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, T_max=max(1, GLOBAL_STEPS), eta_min=2e-5
)
T_DISTILL = 1.5

def distill_loss(slog, tlog, mask, sact, tact):
    s = slog.masked_fill(~mask, -1e4) / T_DISTILL
    t = tlog.masked_fill(~mask, -1e4) / T_DISTILL
    tp = torch.softmax(t.detach(), -1)
    kl = (
        tp * (torch.log_softmax(t.detach(), -1) - torch.log_softmax(s, -1))
    ).sum(-1).mean() * (T_DISTILL*T_DISTILL)
    ce = F.cross_entropy(
        slog.masked_fill(~mask, -1e4),
        tlog.masked_fill(~mask, -1e4).argmax(-1),
    )
    ap = torch.softmax(tact.detach()/T_DISTILL, -1)
    akl = (
        ap * (
            torch.log_softmax(tact.detach()/T_DISTILL, -1)
            - torch.log_softmax(sact/T_DISTILL, -1)
        )
    ).sum(-1).mean() * (T_DISTILL*T_DISTILL)
    return kl + 0.20*ce + 0.05*akl

@torch.no_grad()
def validate_pair(student, limit=128):
    teacher.eval(); student.eval()
    total = agree = 0; js = []
    for off in range(0, min(limit, len(val_items)), BATCH):
        n = min(BATCH, min(limit, len(val_items))-off)
        b = batch(val_items, off, n)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b); sl, _ = student(**b)
        m = b["marker_mask"].bool()
        tm = tl.float().masked_fill(~m, -1e9)
        sm = sl.float().masked_fill(~m, -1e9)
        tp = torch.softmax(tm, -1); sp = torch.softmax(sm, -1)
        agree += int((tm.argmax(-1) == sm.argmax(-1)).sum())
        total += int(tm.shape[0])
        mid = 0.5*(tp+sp)
        j = 0.5*(tp*(tp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        j += 0.5*(sp*(sp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        js.extend(j.cpu().tolist())
    return {"agreement": agree/max(1,total), "mean_js": float(np.mean(js))}

best_state = None; best_score = float("inf")
eval_every = max(40, GLOBAL_STEPS//8)

for step in range(GLOBAL_STEPS):
    b = batch(train_items, step*BATCH + 31)
    opt.zero_grad(set_to_none=True)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
        tl, ta = teacher(**b)
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        sl, sa = student_full(**b)
        loss = distill_loss(sl, tl, b["marker_mask"].bool(), sa, ta)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(delta_params + head_params, 1.0)
    opt.step(); sched.step()

    if step == 0 or (step+1) % eval_every == 0 or step+1 == GLOBAL_STEPS:
        metrics = validate_pair(student_full, limit=96)
        score = metrics["mean_js"] + 0.35*(1.0-metrics["agreement"])
        print(
            f"step={step+1:03d}/{GLOBAL_STEPS} loss={float(loss):.4f} "
            f"agreement={metrics['agreement']:.2%} JS={metrics['mean_js']:.5f}"
        )
        if score < best_score:
            best_score = score
            best_state = {
                n:p.detach().cpu().clone()
                for n,p in student_full.named_parameters() if p.requires_grad
            }

if best_state:
    with torch.no_grad():
        for n,p in student_full.named_parameters():
            if n in best_state:
                p.copy_(best_state[n].to(device=p.device, dtype=p.dtype))

student_full.eval().requires_grad_(False)
v24_full_quality = validate_pair(student_full, limit=min(160, len(val_items)))
print("V2.4 full-head quality:", v24_full_quality)


In [ ]:
#@title 7. Benchmark helper — latency + CUDA activation peak
def _sync():
    torch.cuda.synchronize()

@torch.inference_mode()
def _forward(model, b):
    model.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        return model(**b)

def benchmark_model(model, probe, warmup=3, repeats=10):
    for _ in range(warmup):
        _forward(model, probe)
    _sync()
    start_alloc = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    samples = []
    for _ in range(repeats):
        _sync()
        t0 = time.perf_counter()
        _forward(model, probe)
        _sync()
        samples.append((time.perf_counter()-t0)*1000.0)
    peak = torch.cuda.max_memory_allocated()
    return {
        "median_ms": float(np.median(samples)),
        "p95_ms": float(np.percentile(samples, 95)),
        "activation_peak_delta_mb": float(max(0, peak-start_alloc) / 2**20),
        "total_parameters": int(sum(p.numel() for p in model.parameters())),
    }

probe = batch(val_items, 0, min(BATCH, 3))
teacher_perf = benchmark_model(teacher, probe)
v24_full_perf = benchmark_model(student_full, probe)
v24_full_report = {
    "quality": v24_full_quality,
    "teacher": teacher_perf,
    "student": v24_full_perf,
    "speedup_vs_teacher": teacher_perf["median_ms"] / max(v24_full_perf["median_ms"], 1e-9),
    "sequence_length": int(probe["input_ids"].shape[1]),
    "batch_size": int(probe["input_ids"].shape[0]),
    "new_delta_core_parameters": int(sum(p.numel() for p in v24_core_parameters(student_full))),
}
print(json.dumps(v24_full_report, indent=2))


In [ ]:
#@title 8. Optional compact marker-only decision head
class CompactMarkerDecisionModel(nn.Module):
    def __init__(self, source_model, compact_dim=256, layers=2):
        super().__init__()
        self.encoder = source_model.encoder
        self.type_emb = copy.deepcopy(source_model.type_emb)
        hidden = int(self.encoder.config.hidden_size)
        n_act = int(source_model.act_head[-1].out_features)

        self.in_proj = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, compact_dim, bias=False),
        )
        enc_layer = nn.TransformerEncoderLayer(
            compact_dim,
            nhead=max(1, compact_dim//64),
            dim_feedforward=2*compact_dim,
            dropout=0.05,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.marker_mixer = nn.TransformerEncoder(
            enc_layer, num_layers=layers, enable_nested_tensor=False
        )
        self.scorer = nn.Sequential(
            nn.LayerNorm(compact_dim),
            nn.Linear(compact_dim, compact_dim),
            nn.GELU(),
            nn.Linear(compact_dim, 1),
        )
        self.act_head = nn.Sequential(
            nn.Linear(compact_dim + 4, 128),
            nn.GELU(),
            nn.Linear(128, n_act),
        )

    def forward(self, input_ids, attention_mask, marker_pos, marker_mask, qtype):
        h = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        h = h + self.type_emb(qtype)[:, None, :]

        idx = marker_pos.clamp_min(0)[:, :, None].expand(-1, -1, h.size(-1))
        marker_hidden = torch.gather(h, 1, idx)
        small = torch.cat((h[:, :1], marker_hidden), dim=1)
        small_mask = torch.cat(
            (torch.ones_like(marker_mask[:, :1], dtype=torch.bool), marker_mask.bool()),
            dim=1,
        )

        z = self.in_proj(small)
        z = self.marker_mixer(z, src_key_padding_mask=~small_mask)
        option_z = z[:, 1:]
        logits = self.scorer(option_z).squeeze(-1).float()
        logits = logits.masked_fill(~marker_mask.bool(), -1e4)

        p = torch.softmax(logits.detach(), dim=-1)
        k = marker_mask.sum(-1).clamp(min=2).float()
        ent = -(p * torch.log(p.clamp_min(1e-9))).sum(-1) / torch.log(k)
        if p.shape[-1] >= 2:
            top2 = p.topk(2, dim=-1).values
        else:
            top1 = p.topk(1, dim=-1).values
            top2 = torch.cat((top1, torch.zeros_like(top1)), dim=-1)
        features = torch.stack(
            (top2[:,0], top2[:,0]-top2[:,1], ent, k/255.0), dim=-1
        )
        act_logits = self.act_head(torch.cat((z[:,0].float(), features), dim=-1))
        return logits, act_logits


def train_compact_model():
    compact = CompactMarkerDecisionModel(
        student_full, compact_dim=COMPACT_DIM, layers=COMPACT_LAYERS
    ).to(device)

    # Phase 1: compact head only.
    for p in compact.encoder.parameters(): p.requires_grad = False
    for p in compact.type_emb.parameters(): p.requires_grad = False
    head = [
        p for n,p in compact.named_parameters()
        if not n.startswith("encoder.") and not n.startswith("type_emb.")
    ]
    opt = torch.optim.AdamW(head, lr=6e-4, weight_decay=1e-3)

    best = None; best_score = float("inf")
    eval_every = max(40, COMPACT_STEPS//6)
    for step in range(COMPACT_STEPS):
        b = batch(train_items, step*BATCH + 101)
        opt.zero_grad(set_to_none=True)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, ta = teacher(**b)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            sl, sa = compact(**b)
            loss = distill_loss(sl, tl, b["marker_mask"].bool(), sa, ta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head, 1.0)
        opt.step()

        if step == 0 or (step+1) % eval_every == 0 or step+1 == COMPACT_STEPS:
            m = validate_pair(compact, limit=96)
            score = m["mean_js"] + 0.35*(1-m["agreement"])
            print(
                f"compact-head step={step+1:03d}/{COMPACT_STEPS} "
                f"loss={float(loss):.4f} agreement={m['agreement']:.2%} JS={m['mean_js']:.5f}"
            )
            if score < best_score:
                best_score = score
                best = {
                    n:p.detach().cpu().clone()
                    for n,p in compact.named_parameters() if p.requires_grad
                }

    if best:
        with torch.no_grad():
            for n,p in compact.named_parameters():
                if n in best:
                    p.copy_(best[n].to(device=p.device, dtype=p.dtype))

    # Phase 2: short low-LR joint tune of Delta cores + compact head.
    delta = v24_core_parameters(compact)
    for p in delta: p.requires_grad = True
    joint_head = [
        p for n,p in compact.named_parameters()
        if not n.startswith("encoder.") and p.requires_grad
    ]
    opt = torch.optim.AdamW([
        {"params": delta, "lr": 4e-4, "weight_decay": 1e-3},
        {"params": joint_head, "lr": 1.5e-4, "weight_decay": 1e-3},
    ])

    for step in range(COMPACT_JOINT_STEPS):
        b = batch(train_items, step*BATCH + 401)
        opt.zero_grad(set_to_none=True)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, ta = teacher(**b)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            sl, sa = compact(**b)
            loss = distill_loss(sl, tl, b["marker_mask"].bool(), sa, ta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(delta + joint_head, 1.0)
        opt.step()

        if step == 0 or (step+1) % max(30, COMPACT_JOINT_STEPS//4) == 0 or step+1 == COMPACT_JOINT_STEPS:
            m = validate_pair(compact, limit=96)
            print(
                f"compact-joint step={step+1:03d}/{COMPACT_JOINT_STEPS} "
                f"loss={float(loss):.4f} agreement={m['agreement']:.2%} JS={m['mean_js']:.5f}"
            )

    compact.eval().requires_grad_(False)
    return compact

compact_student = train_compact_model() if RUN_COMPACT_HEAD else None


In [ ]:
#@title 9. Final teacher vs V2.4 comparison
def quality_detail(student, limit=160):
    teacher.eval(); student.eval()
    total = agree = top2 = action_total = action_agree = 0
    js_values = []
    for off in range(0, min(limit, len(val_items)), BATCH):
        n = min(BATCH, min(limit, len(val_items))-off)
        b = batch(val_items, off, n)
        tl, ta = _forward(teacher, b)
        sl, sa = _forward(student, b)

        mm = b["marker_mask"].bool()
        tm = tl.float().masked_fill(~mm, -1e9)
        sm = sl.float().masked_fill(~mm, -1e9)
        tp = torch.softmax(tm, -1); sp = torch.softmax(sm, -1)
        tc = tm.argmax(-1); sc = sm.argmax(-1)
        total += int(tc.numel())
        agree += int((tc == sc).sum())
        k2 = min(2, sm.shape[-1])
        top2 += int((torch.topk(sm, k=k2, dim=-1).indices == tc[:,None]).any(-1).sum())

        mid = 0.5*(tp+sp)
        js = 0.5*(tp*(tp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        js += 0.5*(sp*(sp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        js_values.extend(js.cpu().tolist())

        action_total += int(ta.shape[0])
        action_agree += int((ta.float().argmax(-1) == sa.float().argmax(-1)).sum())

    return {
        "decision_agreement": agree/max(1,total),
        "teacher_top1_in_student_top2": top2/max(1,total),
        "action_head_agreement": action_agree/max(1,action_total),
        "mean_js_divergence": float(np.mean(js_values)),
        "p95_js_divergence": float(np.percentile(js_values, 95)),
    }

models = {"teacher": teacher, "v24_full_head": student_full}
if compact_student is not None:
    models["v24_compact"] = compact_student

report = {
    "architecture": "integrated_memory_v24_bigated_delta_lite",
    "source_model": SOURCE_MODEL,
    "full_attention_layers_replaced": full_layers,
    "sequence_length": int(probe["input_ids"].shape[1]),
    "batch_size": int(probe["input_ids"].shape[0]),
    "models": {},
}

teacher_perf = benchmark_model(teacher, probe)
report["models"]["teacher"] = {
    "performance": teacher_perf,
    "quality_vs_teacher": {"decision_agreement": 1.0, "mean_js_divergence": 0.0},
}

for name, model in models.items():
    if name == "teacher": continue
    perf = benchmark_model(model, probe)
    quality = quality_detail(model, limit=min(160, len(val_items)))
    report["models"][name] = {
        "performance": perf,
        "quality_vs_teacher": quality,
        "speedup_vs_teacher": teacher_perf["median_ms"]/max(perf["median_ms"], 1e-9),
        "parameter_ratio_vs_teacher": perf["total_parameters"]/teacher_perf["total_parameters"],
        "gates": {
            "decision_agreement_ge_95pct": quality["decision_agreement"] >= 0.95,
            "mean_js_le_0_020": quality["mean_js_divergence"] <= 0.020,
            "faster_than_teacher": perf["median_ms"] < teacher_perf["median_ms"],
        },
    }

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT_DIR / "v24_bigated_delta_lite_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n=== V2.4 FINAL COMPARISON ===")
for name, result in report["models"].items():
    p = result["performance"]; q = result["quality_vs_teacher"]
    print(f"\n{name}")
    print(f"  params       : {p['total_parameters']:,}")
    print(f"  latency      : {p['median_ms']:.2f} ms median | {p['p95_ms']:.2f} ms p95")
    print(f"  act. peak Δ  : {p['activation_peak_delta_mb']:.1f} MiB")
    if name != "teacher":
        print(f"  speedup      : {result['speedup_vs_teacher']:.2f}x")
        print(f"  agreement    : {q['decision_agreement']:.2%}")
        print(f"  top-2 recall : {q['teacher_top1_in_student_top2']:.2%}")
        print(f"  action agree : {q['action_head_agreement']:.2%}")
        print(f"  mean / p95 JS: {q['mean_js_divergence']:.5f} / {q['p95_js_divergence']:.5f}")
        print(f"  gates        : {result['gates']}")

print("\nsaved:", report_path)


## Interpretation

The full-head result isolates the Bi-Gated-Delta replacement. The compact result then isolates the benefit of processing only CLS + option markers.

A strong result should satisfy all three gates: at least 95% decision agreement, mean JS at most 0.020, and lower median latency than the teacher. If the full-head version has high fidelity but insufficient speed, the next ablation should reduce reverse scans or retain only a small number of native full-attention layers. If the compact version is much faster but loses fidelity, keep the V2.4 encoder and tune only compact-head capacity/training.
